# Step 1 baseline assessment area extents

This notebook calculates the mapped extent of Jamaica's baseline land-cover categories, protected-area intersections, and selected marine ecosystems for Paper 3 Figure 1 and the accompanying text.

## Method

- Calculate areas in `EPSG:3448`, the Jamaica metric grid.
- Use the 2013 land-use/land-cover layer for terrestrial classes, except mangroves.
- Remove the 2013 `Mangrove Forest` class, erase Forces of Nature mangrove areas from the remaining land-cover polygons, then add the dissolved Forces of Nature mangrove layer back as `Mangrove`.
- Dissolve forest reserves and protected areas before clipping so overlaps are not double-counted.
- Save CSV outputs to `dphil_paper_3/processed_data/baseline_assessment/area_extents`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.max_rows = 100
pd.options.display.max_columns = 50

METRIC_CRS = 'EPSG:3448'


In [ ]:
def find_dphil_root(start_path=None):
    search_start = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate_path in [search_start, *search_start.parents]:
        if (candidate_path / 'dphil_paper_3').exists() and (candidate_path / 'dphil_common_cross_cutting').exists():
            return candidate_path
        nested_root = candidate_path / 'dphil_papers'
        if (nested_root / 'dphil_paper_3').exists() and (nested_root / 'dphil_common_cross_cutting').exists():
            return nested_root
    raise FileNotFoundError('Could not locate dphil_papers root from the current working directory.')


dphil_root = find_dphil_root()
paper_root = dphil_root / 'dphil_paper_3'
common_root = dphil_root / 'dphil_common_cross_cutting'
output_dir = paper_root / 'processed_data' / 'baseline_assessment' / 'area_extents'
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    'landcover_2013': common_root / 'common_incoming_data' / 'landcover' / '2013_landcover' / '2013_landuse_LandCover.shp',
    'mangroves_fon': paper_root / 'inputs' / 'forces_of_nature_mangroves' / 'mangroves.shp',
    'forest_reserves': common_root / 'common_incoming_data' / 'protected_landcover' / 'Forest_reserves.shp',
    'protected_areas': common_root / 'common_incoming_data' / 'protected_landcover' / 'Protected_areas.shp',
    'coral_reefs': paper_root / 'processed_data' / 'corals' / 'corals_clipped_1000m.shp',
    'seagrass': paper_root / 'processed_data' / 'seagrass' / 'seagrass_clipped_10000m.shp',
}

missing_paths = {name: path for name, path in paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(missing_paths)

display(pd.DataFrame({'layer': paths.keys(), 'path': [str(path) for path in paths.values()]}))


In [ ]:
landcover_category_mapping = {
    'Bare Rock': 'Bare Rock',
    'Fields: Herbaceous crops, fallow, cultivated vegetables': 'Agriculture',
    'Fields: Pasture,Human disturbed, grassland': 'Agriculture',
    'Herbaceous Wetland': 'Freshwater wetland',
    'Mangrove Forest': 'Mangrove',
    'Fields: Bare Land': 'Agriculture',
    'Open dry forest - Short': 'Open dry forest',
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest',
    'Plantation: Tree crops, shrub crops, sugar cane, banana': 'Plantation',
    'Quarry': 'Bauxite extraction / quarry',
    'Water Body': 'Water body',
    'Buildings and other infrastructures': 'Buildings and other infrastructure',
    'Fields and Secondary Forest': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Bamboo and Fields': 'Mixed land use: agriculture and bamboo',
    'Bauxite Extraction': 'Bauxite extraction / quarry',
    'Disturbed broadleaved forest (Secondary Forest)': 'Forest',
    'Fields  and Bamboo': 'Mixed land use: agriculture and bamboo',
    'Bamboo and Secondary Forest': 'Mixed land use: agriculture and bamboo',
    'Hardwood Plantation: Euculytus': 'Plantation',
    'Hardwood Plantation: Mixed': 'Plantation',
    'Swamp Forest': 'Swamp forest',
    'Fields or Secondary Forest/Pine Plantation': 'Mixed land use: forests with bamboo or agriculture/plantation',
    'Hardwood Plantation: Mahoe': 'Plantation',
    'Hardwood Plantation: Mahogany': 'Plantation',
    'Bamboo': 'Bamboo',
    'Closed broadleaved forest (Primary Forest)': 'Forest',
    'Secondary Forest': 'Forest',
}

# Keep the paper categories identical to the mapped land-cover categories.
paper_category_mapping = {category: category for category in sorted(set(landcover_category_mapping.values()))}


In [ ]:
def load_valid_layer(path, metric_crs=METRIC_CRS):
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f'Layer has no CRS: {path}')
    layer = layer.to_crs(metric_crs)
    layer = layer[layer.geometry.notna()].copy()
    layer['geometry'] = layer.geometry.make_valid()
    return layer[~layer.geometry.is_empty].copy()


def add_area_columns(summary, reference_area_m2=None):
    summary = summary.copy()
    summary['area_km2'] = summary['area_m2'] / 1e6
    summary['area_ha'] = summary['area_m2'] / 1e4
    summary['share_of_terrestrial_area_percent'] = (
        np.nan if reference_area_m2 is None else summary['area_m2'] / reference_area_m2 * 100
    )
    summary['protected_area_m2'] = summary.get('protected_area_m2', 0.0)
    summary['protected_area_km2'] = summary['protected_area_m2'] / 1e6
    summary['protected_area_ha'] = summary['protected_area_m2'] / 1e4
    summary['percentage_protected'] = np.where(
        summary['area_m2'] > 0,
        summary['protected_area_m2'] / summary['area_m2'] * 100,
        np.nan,
    )
    return summary


def summarize_area_and_protection(total_gdf, protected_gdf, group_column, reference_area_m2):
    total = total_gdf.groupby(group_column, as_index=False)['area_m2'].sum()
    protected = protected_gdf.groupby(group_column, as_index=False)['protected_area_m2'].sum()
    summary = total.merge(protected, on=group_column, how='left').fillna({'protected_area_m2': 0.0})
    return add_area_columns(summary, reference_area_m2).sort_values('area_m2', ascending=False).reset_index(drop=True)


def display_area_table(summary, category_column):
    columns = [
        category_column,
        'area_km2',
        'area_ha',
        'share_of_terrestrial_area_percent',
        'protected_area_km2',
        'protected_area_ha',
        'percentage_protected',
    ]
    display(summary[columns].round(2))



def dissolve_to_single_feature(layer):
    return gpd.GeoDataFrame(geometry=[layer.geometry.union_all()], crs=METRIC_CRS)


In [ ]:
landcover_2013 = load_valid_layer(paths['landcover_2013']).dropna(subset=['Classify']).copy()
unknown_classes = sorted(set(landcover_2013['Classify']) - set(landcover_category_mapping))
if unknown_classes:
    raise ValueError(f'Unmapped land-cover classes: {unknown_classes}')

mangroves_fon = load_valid_layer(paths['mangroves_fon'])
forest_reserves = load_valid_layer(paths['forest_reserves'])
protected_areas = load_valid_layer(paths['protected_areas'])
coral_reefs = load_valid_layer(paths['coral_reefs'])
seagrass = load_valid_layer(paths['seagrass'])

original_2013_landcover_area_m2 = landcover_2013.geometry.area.sum()
original_2013_mangrove_area_m2 = landcover_2013.loc[
    landcover_2013['Classify'].eq('Mangrove Forest')
].geometry.area.sum()


In [ ]:
mangroves_fon_geometry = mangroves_fon.geometry.union_all()
mangroves_fon_landcover = gpd.GeoDataFrame(
    {
        'source_class': ['Forces of Nature mangroves'],
        'category': ['Mangrove'],
        'paper_category': [paper_category_mapping['Mangrove']],
    },
    geometry=[mangroves_fon_geometry],
    crs=METRIC_CRS,
)

landcover_without_2013_mangrove = landcover_2013[~landcover_2013['Classify'].eq('Mangrove Forest')].copy()
intersects_fon_mangroves = landcover_without_2013_mangrove.geometry.intersects(mangroves_fon_geometry)
landcover_without_2013_mangrove.loc[intersects_fon_mangroves, 'geometry'] = (
    landcover_without_2013_mangrove.loc[intersects_fon_mangroves].geometry.difference(mangroves_fon_geometry)
)
landcover_without_2013_mangrove['geometry'] = landcover_without_2013_mangrove.geometry.make_valid()
landcover_without_2013_mangrove = landcover_without_2013_mangrove[
    landcover_without_2013_mangrove.geometry.notna() & ~landcover_without_2013_mangrove.geometry.is_empty
].copy()

landcover_without_2013_mangrove['source_class'] = landcover_without_2013_mangrove['Classify']
landcover_without_2013_mangrove['category'] = landcover_without_2013_mangrove['Classify'].replace(landcover_category_mapping)
landcover_without_2013_mangrove['paper_category'] = landcover_without_2013_mangrove['category'].map(paper_category_mapping)

landcover = gpd.GeoDataFrame(
    pd.concat(
        [
            landcover_without_2013_mangrove[['source_class', 'category', 'paper_category', 'geometry']],
            mangroves_fon_landcover[['source_class', 'category', 'paper_category', 'geometry']],
        ],
        ignore_index=True,
    ),
    geometry='geometry',
    crs=METRIC_CRS,
)
landcover['area_m2'] = landcover.geometry.area

total_terrestrial_area_m2 = landcover['area_m2'].sum()
fon_mangrove_area_m2 = mangroves_fon_landcover.geometry.area.sum()

print(f'Original 2013 land-cover area: {original_2013_landcover_area_m2 / 1e6:,.2f} km²')
print(f'2013 mangrove class removed: {original_2013_mangrove_area_m2 / 1e6:,.2f} km²')
print(f'Forces of Nature mangroves added: {fon_mangrove_area_m2 / 1e6:,.2f} km²')
print(f'Final mapped terrestrial extent: {total_terrestrial_area_m2 / 1e6:,.2f} km²')


In [ ]:
protected_raw = gpd.GeoDataFrame(
    pd.concat([forest_reserves[['geometry']], protected_areas[['geometry']]], ignore_index=True),
    geometry='geometry',
    crs=METRIC_CRS,
)
combined_protected_layers = gpd.GeoDataFrame(
    geometry=[protected_raw.geometry.union_all()],
    crs=METRIC_CRS,
)

protected_landcover = gpd.clip(
    landcover[['source_class', 'category', 'paper_category', 'geometry']].copy(),
    combined_protected_layers,
)
protected_landcover['protected_area_m2'] = protected_landcover.geometry.area

protected_terrestrial_area_m2 = protected_landcover['protected_area_m2'].sum()
raw_protected_area_m2 = protected_raw.geometry.area.sum()
dissolved_protected_area_m2 = combined_protected_layers.geometry.area.sum()

print(f'Terrestrial area inside dissolved protected network: {protected_terrestrial_area_m2 / 1e6:,.2f} km²')
print(f'Raw/dissolved protected-area ratio: {raw_protected_area_m2 / dissolved_protected_area_m2:.2f}')


In [ ]:
source_class_summary = summarize_area_and_protection(
    landcover,
    protected_landcover,
    'source_class',
    total_terrestrial_area_m2,
).rename(columns={'source_class': 'source_landcover_class'})

figure1_category_summary = summarize_area_and_protection(
    landcover,
    protected_landcover,
    'category',
    total_terrestrial_area_m2,
)

paper_category_summary = summarize_area_and_protection(
    landcover,
    protected_landcover,
    'paper_category',
    total_terrestrial_area_m2,
)

if (figure1_category_summary['protected_area_m2'] > figure1_category_summary['area_m2'] + 1e-6).any():
    raise AssertionError('At least one category has protected area greater than total area.')

direct_protected_percent = protected_terrestrial_area_m2 / total_terrestrial_area_m2 * 100
weighted_protected_percent = (
    figure1_category_summary['percentage_protected'] * figure1_category_summary['area_m2']
).sum() / figure1_category_summary['area_m2'].sum()
if not np.isclose(weighted_protected_percent, direct_protected_percent):
    raise AssertionError('Weighted and direct protected-area percentages do not match.')

print('Paper text categories')
display_area_table(paper_category_summary, 'paper_category')


## Forest type breakdown

This table reports total forest including open dry forest, then breaks that total into primary forest, secondary forest, and open dry forest tall/short. It keeps mangrove and swamp forest separate, consistent with the baseline category mapping.

In [ ]:
forest_type_mapping = {
    'Closed broadleaved forest (Primary Forest)': 'Primary forest',
    'Disturbed broadleaved forest (Secondary Forest)': 'Secondary forest',
    'Secondary Forest': 'Secondary forest',
    'Open dry forest - Tall (Woodland/Savanna)': 'Open dry forest - tall',
    'Open dry forest - Short': 'Open dry forest - short',
}
forest_type_order = [
    'Total forest including open dry forest',
    'Primary forest',
    'Secondary forest',
    'Open dry forest - tall',
    'Open dry forest - short',
]

forest_landcover = landcover[landcover['source_class'].isin(forest_type_mapping)].copy()
forest_landcover['forest_type'] = forest_landcover['source_class'].map(forest_type_mapping)

protected_forest_landcover = protected_landcover[
    protected_landcover['source_class'].isin(forest_type_mapping)
].copy()
protected_forest_landcover['forest_type'] = protected_forest_landcover['source_class'].map(forest_type_mapping)

forest_component_summary = summarize_area_and_protection(
    forest_landcover,
    protected_forest_landcover,
    'forest_type',
    total_terrestrial_area_m2,
)

forest_total_row = add_area_columns(
    pd.DataFrame([{
        'forest_type': 'Total forest including open dry forest',
        'area_m2': forest_component_summary['area_m2'].sum(),
        'protected_area_m2': forest_component_summary['protected_area_m2'].sum(),
    }]),
    reference_area_m2=total_terrestrial_area_m2,
)

forest_type_summary = pd.concat(
    [forest_total_row, forest_component_summary],
    ignore_index=True,
)
forest_type_summary['forest_type'] = pd.Categorical(
    forest_type_summary['forest_type'],
    categories=forest_type_order,
    ordered=True,
)
forest_type_summary = forest_type_summary.sort_values('forest_type').reset_index(drop=True)
forest_type_summary['forest_type'] = forest_type_summary['forest_type'].astype(str)
forest_type_summary['share_of_total_forest_percent'] = np.where(
    forest_type_summary['forest_type'].eq('Total forest including open dry forest'),
    100.0,
    forest_type_summary['area_m2'] / forest_total_row['area_m2'].iloc[0] * 100,
)

forest_display_columns = [
    'forest_type',
    'area_km2',
    'area_ha',
    'share_of_total_forest_percent',
    'share_of_terrestrial_area_percent',
    'protected_area_km2',
    'protected_area_ha',
    'percentage_protected',
]
display(forest_type_summary[forest_display_columns].round(2))


In [ ]:
coral_reefs_dissolved = dissolve_to_single_feature(coral_reefs)
seagrass_dissolved = dissolve_to_single_feature(seagrass)
protected_coral_reefs = gpd.clip(coral_reefs_dissolved, combined_protected_layers)
protected_seagrass = gpd.clip(seagrass_dissolved, combined_protected_layers)

marine_summary = pd.DataFrame([
    {
        'domain': 'marine',
        'category': 'Coral reefs',
        'area_m2': coral_reefs_dissolved.geometry.area.sum(),
        'protected_area_m2': protected_coral_reefs.geometry.area.sum(),
    },
    {
        'domain': 'marine',
        'category': 'Seagrass',
        'area_m2': seagrass_dissolved.geometry.area.sum(),
        'protected_area_m2': protected_seagrass.geometry.area.sum(),
    },
])
marine_summary = add_area_columns(marine_summary, reference_area_m2=None)

print('Marine ecosystem categories')
display_area_table(marine_summary, 'category')


In [ ]:
protected_network_summary = pd.DataFrame([{
    'original_2013_landcover_area_km2': original_2013_landcover_area_m2 / 1e6,
    'total_terrestrial_area_km2': total_terrestrial_area_m2 / 1e6,
    'total_terrestrial_area_ha': total_terrestrial_area_m2 / 1e4,
    'landcover_2013_mangrove_area_km2': original_2013_mangrove_area_m2 / 1e6,
    'landcover_2013_mangrove_area_ha': original_2013_mangrove_area_m2 / 1e4,
    'forces_of_nature_mangrove_area_km2': fon_mangrove_area_m2 / 1e6,
    'forces_of_nature_mangrove_area_ha': fon_mangrove_area_m2 / 1e4,
    'terrestrial_protected_intersection_km2': protected_terrestrial_area_m2 / 1e6,
    'terrestrial_protected_intersection_ha': protected_terrestrial_area_m2 / 1e4,
    'terrestrial_protected_intersection_percent': direct_protected_percent,
    'raw_protected_layer_area_km2': raw_protected_area_m2 / 1e6,
    'dissolved_protected_layer_area_km2': dissolved_protected_area_m2 / 1e6,
    'raw_to_dissolved_protected_area_ratio': raw_protected_area_m2 / dissolved_protected_area_m2,
}])

combined_ecosystem_summary = pd.concat(
    [figure1_category_summary.assign(domain='terrestrial'), marine_summary],
    ignore_index=True,
    sort=False,
)[[
    'domain',
    'category',
    'area_m2',
    'area_km2',
    'area_ha',
    'share_of_terrestrial_area_percent',
    'protected_area_m2',
    'protected_area_km2',
    'protected_area_ha',
    'percentage_protected',
]]

source_class_summary.to_csv(output_dir / 'landcover_source_class_area_extents.csv', index=False, float_format='%.6f')
figure1_category_summary.to_csv(output_dir / 'figure1_terrestrial_category_area_extents.csv', index=False, float_format='%.6f')
paper_category_summary.to_csv(output_dir / 'paper_text_terrestrial_category_area_extents.csv', index=False, float_format='%.6f')
forest_type_summary.to_csv(output_dir / 'forest_type_area_extents.csv', index=False, float_format='%.6f')
paper_category_summary.to_csv(output_dir / 'paper_text_terrestrial_aggregate_area_extents.csv', index=False, float_format='%.6f')
marine_summary.to_csv(output_dir / 'marine_ecosystem_area_extents.csv', index=False, float_format='%.6f')
combined_ecosystem_summary.to_csv(output_dir / 'ecosystem_area_extents_with_protected_status.csv', index=False, float_format='%.6f')
protected_network_summary.to_csv(output_dir / 'protected_network_area_summary.csv', index=False, float_format='%.6f')

print(f'Outputs saved to: {output_dir}')


In [ ]:
def get_summary_row(summary_table, category_column, category_name):
    matching_rows = summary_table[summary_table[category_column].eq(category_name)]
    if matching_rows.empty:
        raise ValueError(f'Missing category: {category_name}')
    return matching_rows.iloc[0]


def build_extent_record(display_category, source_row, category_column):
    return {
        'display_category': display_category,
        'source_category': source_row[category_column],
        'area_km2': source_row['area_km2'],
        'area_ha': source_row['area_ha'],
        'share_of_terrestrial_area_percent': source_row.get('share_of_terrestrial_area_percent', np.nan),
        'protected_area_km2': source_row.get('protected_area_km2', np.nan),
        'protected_area_ha': source_row.get('protected_area_ha', np.nan),
        'percentage_protected': source_row.get('percentage_protected', np.nan),
    }


forest_total = get_summary_row(
    forest_type_summary,
    'forest_type',
    'Total forest including open dry forest',
)
primary_forest = get_summary_row(forest_type_summary, 'forest_type', 'Primary forest')
secondary_forest = get_summary_row(forest_type_summary, 'forest_type', 'Secondary forest')
open_dry_forest_tall = get_summary_row(forest_type_summary, 'forest_type', 'Open dry forest - tall')
open_dry_forest_short = get_summary_row(forest_type_summary, 'forest_type', 'Open dry forest - short')

mixed_forest_bamboo_agriculture = get_summary_row(
    paper_category_summary,
    'paper_category',
    'Mixed land use: forests with bamboo or agriculture/plantation',
)
mixed_agriculture_bamboo = get_summary_row(
    paper_category_summary,
    'paper_category',
    'Mixed land use: agriculture and bamboo',
)
mixed_land_use = mixed_forest_bamboo_agriculture.copy()
mixed_land_use['paper_category'] = 'Mixed land use'
mixed_land_use['area_m2'] = mixed_forest_bamboo_agriculture['area_m2'] + mixed_agriculture_bamboo['area_m2']
mixed_land_use['protected_area_m2'] = (
    mixed_forest_bamboo_agriculture['protected_area_m2'] + mixed_agriculture_bamboo['protected_area_m2']
)
mixed_land_use = add_area_columns(
    pd.DataFrame([mixed_land_use]),
    reference_area_m2=total_terrestrial_area_m2,
).iloc[0]

baseline_extent_records = [
    build_extent_record('Forest including primary, secondary and open dry forest', forest_total, 'forest_type'),
    build_extent_record('Mixed land use', mixed_land_use, 'paper_category'),
    build_extent_record('Agriculture', get_summary_row(paper_category_summary, 'paper_category', 'Agriculture'), 'paper_category'),
    build_extent_record(
        'Buildings and other infrastructure',
        get_summary_row(paper_category_summary, 'paper_category', 'Buildings and other infrastructure'),
        'paper_category',
    ),
    build_extent_record('Plantation', get_summary_row(paper_category_summary, 'paper_category', 'Plantation'), 'paper_category'),
    build_extent_record('Mangroves', get_summary_row(paper_category_summary, 'paper_category', 'Mangrove'), 'paper_category'),
    build_extent_record(
        'Freshwater wetlands',
        get_summary_row(paper_category_summary, 'paper_category', 'Freshwater wetland'),
        'paper_category',
    ),
    build_extent_record('Water bodies', get_summary_row(paper_category_summary, 'paper_category', 'Water body'), 'paper_category'),
    build_extent_record('Swamp forest', get_summary_row(paper_category_summary, 'paper_category', 'Swamp forest'), 'paper_category'),
    build_extent_record('Coral reefs', get_summary_row(marine_summary, 'category', 'Coral reefs'), 'category'),
    build_extent_record('Seagrass', get_summary_row(marine_summary, 'category', 'Seagrass'), 'category'),
]

manuscript_baseline_extents = pd.DataFrame(baseline_extent_records)

forest_share_of_protected_terrestrial_area_percent = (
    forest_total['protected_area_km2']
    / protected_network_summary['terrestrial_protected_intersection_km2'].iloc[0]
    * 100
)

manuscript_protection_summary = pd.DataFrame([
    {
        'metric': 'Mapped terrestrial baseline under formal protection',
        'area_km2': protected_network_summary['terrestrial_protected_intersection_km2'].iloc[0],
        'area_ha': protected_network_summary['terrestrial_protected_intersection_ha'].iloc[0],
        'percent': protected_network_summary['terrestrial_protected_intersection_percent'].iloc[0],
    },
    {
        'metric': 'Protected forest area',
        'area_km2': forest_total['protected_area_km2'],
        'area_ha': forest_total['protected_area_ha'],
        'percent': forest_total['percentage_protected'],
    },
    {
        'metric': 'Forest share of protected terrestrial area',
        'area_km2': forest_total['protected_area_km2'],
        'area_ha': forest_total['protected_area_ha'],
        'percent': forest_share_of_protected_terrestrial_area_percent,
    },
])

manuscript_forest_protection = pd.DataFrame([
    build_extent_record('Primary forest', primary_forest, 'forest_type'),
    build_extent_record('Secondary forest', secondary_forest, 'forest_type'),
    build_extent_record('Tall open dry forest', open_dry_forest_tall, 'forest_type'),
    build_extent_record('Short open dry forest', open_dry_forest_short, 'forest_type'),
])

manuscript_wetland_coastal_marine_protection = pd.DataFrame([
    build_extent_record(
        'Freshwater wetlands',
        get_summary_row(paper_category_summary, 'paper_category', 'Freshwater wetland'),
        'paper_category',
    ),
    build_extent_record('Mangroves', get_summary_row(paper_category_summary, 'paper_category', 'Mangrove'), 'paper_category'),
    build_extent_record('Seagrass', get_summary_row(marine_summary, 'category', 'Seagrass'), 'category'),
    build_extent_record('Coral reefs', get_summary_row(marine_summary, 'category', 'Coral reefs'), 'category'),
])

manuscript_baseline_extents.to_csv(
    output_dir / 'manuscript_paragraph_baseline_extents.csv',
    index=False,
    float_format='%.6f',
)
manuscript_protection_summary.to_csv(
    output_dir / 'manuscript_paragraph_protection_summary.csv',
    index=False,
    float_format='%.6f',
)
manuscript_forest_protection.to_csv(
    output_dir / 'manuscript_paragraph_forest_protection.csv',
    index=False,
    float_format='%.6f',
)
manuscript_wetland_coastal_marine_protection.to_csv(
    output_dir / 'manuscript_paragraph_wetland_coastal_marine_protection.csv',
    index=False,
    float_format='%.6f',
)

print('Manuscript paragraph: baseline extents')
display(manuscript_baseline_extents.round(2))
print('Manuscript paragraph: protection summary')
display(manuscript_protection_summary.round(2))
print('Manuscript paragraph: forest protection')
display(manuscript_forest_protection.round(2))
print('Manuscript paragraph: wetland, coastal and marine protection')
display(manuscript_wetland_coastal_marine_protection.round(2))

print('Manuscript-ready rounded values')
print(
    f"Mapped terrestrial baseline area: {total_terrestrial_area_m2 / 1e6:,.0f} km²; "
    f"formal protection: {protected_network_summary['terrestrial_protected_intersection_km2'].iloc[0]:,.0f} km² "
    f"({protected_network_summary['terrestrial_protected_intersection_percent'].iloc[0]:.1f}%)."
)
print(
    f"Forest including open dry forest: {forest_total['area_km2']:,.0f} km² "
    f"({forest_total['share_of_terrestrial_area_percent']:.1f}%); protected forest: "
    f"{forest_total['protected_area_km2']:,.0f} km² "
    f"({forest_total['percentage_protected']:.1f}% of forest; "
    f"{forest_share_of_protected_terrestrial_area_percent:.1f}% of protected terrestrial area)."
)
